# Préparation du dataset - 15000 Gutenberg Books (Kaggle)
Dataset multilingue : on va d'abord explorer sa structure, repérer un éventuel
fichier de métadonnées (langue), filtrer sur le français, puis nettoyer et tokeniser.


## 1. Explorer la structure du dataset

In [ ]:
import os

DATASET_PATH = "/kaggle/input/datasets/mateibejan/15000-gutenberg-books"

all_files = []
for root, dirs, files in os.walk(DATASET_PATH):
    for f in files:
        all_files.append(os.path.join(root, f))

print(f"{len(all_files)} fichiers trouves\n")

# Compte les extensions presentes
from collections import Counter
exts = Counter(os.path.splitext(f)[1].lower() for f in all_files)
print("Extensions trouvees:", dict(exts))

print("\nExemples de fichiers:")
for f in all_files[:15]:
    print(" -", f)

## 2. Repérer un fichier de métadonnées (souvent un .csv avec langue/auteur/titre)

In [ ]:
import pandas as pd

csv_files = [f for f in all_files if f.lower().endswith(".csv")]
print("CSV trouves:", csv_files)

metadata = None
if csv_files:
    metadata = pd.read_csv(csv_files[0])
    print("\nColonnes:", list(metadata.columns))
    print(metadata.head())
else:
    print("Aucun CSV trouve, on filtrera autrement (cellule suivante adaptee au besoin).")

## 3. Filtrer les livres en français
Adapte le nom de colonne (`lang_col`) selon ce qu'affiche la cellule précédente
(souvent `language`, `Language`, ou `lang`). Si la colonne contient des codes
type "fr"/"french", ajuste le filtre en conséquence.

In [ ]:
txt_files = [f for f in all_files if f.lower().endswith(".txt")]

french_files = []

if metadata is not None:
    # Essaie de deviner automatiquement la colonne langue
    lang_col_candidates = [c for c in metadata.columns if "lang" in c.lower()]
    print("Colonnes candidates pour la langue:", lang_col_candidates)

    if lang_col_candidates:
        lang_col = lang_col_candidates[0]
        print("\nValeurs uniques dans", lang_col, ":", metadata[lang_col].unique()[:20])

        # Filtre : adapte "french"/"fr" selon ce que tu vois affiche au-dessus
        mask = metadata[lang_col].astype(str).str.lower().str.contains("french|fr", na=False)
        french_meta = metadata[mask]
        print(f"\n{len(french_meta)} livres francais trouves dans les metadonnees")
        print(french_meta.head())
    else:
        print("Pas de colonne langue evidente, filtrage manuel necessaire.")
else:
    print("Pas de metadata.csv, filtrage manuel necessaire.")

## 4. Sélectionner les fichiers texte correspondants
Une fois `french_meta` obtenu à la cellule précédente, on relie ses entrées
aux fichiers `.txt` réels (souvent via un ID ou un nom de fichier dans une colonne).

In [ ]:
N_BOOKS = 3

# Si on a des metadonnees filtrees en francais, on tente de matcher les fichiers .txt
selected_files = []

if metadata is not None and len(french_meta) > 0:
    # Cherche une colonne qui ressemble a un identifiant/nom de fichier
    id_col_candidates = [c for c in french_meta.columns if any(k in c.lower() for k in ["id", "file", "name", "title"])]
    print("Colonnes candidates pour matcher les fichiers:", id_col_candidates)

    for _, row in french_meta.head(N_BOOKS * 5).iterrows():
        for col in id_col_candidates:
            val = str(row[col])
            match = [f for f in txt_files if val in os.path.basename(f)]
            if match:
                selected_files.append(match[0])
                break
        if len(selected_files) >= N_BOOKS:
            break

print(f"\n{len(selected_files)} fichiers matches automatiquement")
for f in selected_files:
    print(" -", f)

if not selected_files:
    print("\nAucun match automatique. Inspecte manuellement les colonnes ci-dessus")
    print("et les noms de fichiers .txt pour ajuster la logique de matching.")
    print("\nExemples de noms de fichiers .txt disponibles:")
    for f in txt_files[:10]:
        print(" -", os.path.basename(f))

## 5. Charger et nettoyer le texte
Retire le header/footer légal Gutenberg.

In [ ]:
import re

def clean_gutenberg_text(raw: str) -> str:
    start_match = re.search(r"\*\*\*\s*START OF.*?\*\*\*", raw, re.IGNORECASE | re.DOTALL)
    end_match = re.search(r"\*\*\*\s*END OF.*?\*\*\*", raw, re.IGNORECASE | re.DOTALL)

    if start_match:
        raw = raw[start_match.end():]
    if end_match:
        raw = raw[:end_match.start()]

    raw = re.sub(r"\r\n", "\n", raw)
    raw = re.sub(r"\n{3,}", "\n\n", raw)
    return raw.strip()


all_text = []
for f in selected_files:
    with open(f, "r", encoding="utf-8", errors="ignore") as file:
        raw = file.read()
    cleaned = clean_gutenberg_text(raw)
    all_text.append(cleaned)
    print(f"{os.path.basename(f)}: {len(cleaned):,} caracteres apres nettoyage")

full_text = "\n\n".join(all_text)
print(f"\nTotal: {len(full_text):,} caracteres")

## 6. Construire le vocabulaire (tokenisation char-level)

In [ ]:
chars = sorted(list(set(full_text)))
vocab_size = len(chars)
print(f"Vocabulaire: {vocab_size} caracteres uniques")
print("".join(chars))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str):
    return [stoi[c] for c in s]

def decode(tokens):
    return "".join(itos[t] for t in tokens)

test = encode("Bonjour")
print(test, "->", decode(test))

## 7. Split train / validation (90% / 10%) et sauvegarde

In [ ]:
import torch
import pickle

data = torch.tensor(encode(full_text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train: {len(train_data):,} tokens")
print(f"Val:   {len(val_data):,} tokens")

torch.save(train_data, "/kaggle/working/train.pt")
torch.save(val_data, "/kaggle/working/val.pt")

with open("/kaggle/working/vocab.pkl", "wb") as f:
    pickle.dump({"stoi": stoi, "itos": itos, "vocab_size": vocab_size}, f)

print("\nFichiers sauvegardes dans /kaggle/working/")

## 8. Vérification rapide

In [ ]:
train_check = torch.load("/kaggle/working/train.pt")
val_check = torch.load("/kaggle/working/val.pt")

print("Premier extrait du train:")
print(decode(train_check[:300].tolist()))